# GSM8K CoT-SFT → DPO Pipeline
Base GPT-2 → CoT-SFT → DPO 세 단계 학습 및 평가

In [1]:
# 1. GPU 확인
!nvidia-smi

Wed Jun  3 11:40:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# 2. Drive 마운트 및 프로젝트 디렉토리 이동
from google.colab import drive
import os

drive.mount('/content/drive')
%cd /content/drive/MyDrive/서강대학교/NLP_FINAL

print('Working dir:', os.getcwd())
print('Python files:', [f for f in os.listdir('.') if f.endswith('.py')])
print('Data files:', os.listdir('data') if os.path.exists('data') else 'data/ 없음')

Mounted at /content/drive
/content/drive/MyDrive/서강대학교/NLP_FINAL
Working dir: /content/drive/MyDrive/서강대학교/NLP_FINAL
Python files: ['utils.py', 'config.py', 'optimizer.py', 'prepare_submit.py', 'sanity_check.py', 'optimizer_test.py', 'prepare_gsm8k.py', 'paraphrase_detection.py', 'evaluation.py', 'sonnet_generation.py', 'gsm8k_eval.py', 'make_splits.py', 'baseline_gsm8k.py', 'classifier.py', 'reasoning_generation.py', 'gpt_datasets.py', 'eval_sft.py', 'generate_rejected.py', 'train_dpo.py']
Data files: ['ids-cfimdb-dev.csv', 'sonnets_held_out_dev.txt', 'TRUE_sonnets_held_out_dev.txt', 'quora-train.csv', 'quora-test-student.csv', 'quora-dev.csv', 'gsm8k_small_train.txt', 'ids-cfimdb-test-student.csv', 'ids-cfimdb-train.csv', 'sonnets.txt', 'ids-sst-test-student.csv', 'gsm8k_small_held_out.txt', 'sonnets_held_out.txt', 'ids-sst-dev.csv', 'ids-sst-train.csv', 'gsm8k_dev_prompts.txt', 'gsm8k_dev.jsonl', 'gsm8k_dpo_source.jsonl', 'gsm8k_sft_train.txt', 'dpo_pairs.jsonl']


In [4]:
# 3. 의존성 설치
!pip install einops transformers trl datasets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 21.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 22.7 MB/s eta 0:00:00:00:0100:01


In [13]:
# 4. CoT-SFT 학습
!python reasoning_generation.py \
    --use_gpu \
    --epochs 10 \
    --lr 1e-5 \
    --batch_size 8 \
    --reasoning_path data/gsm8k_sft_train.txt \
    --held_out_reasoning_path data/gsm8k_small_held_out.txt

tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 133kB/s]
vocab.json: 100% 1.04M/1.04M [00:00<00:00, 8.28MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 9.77MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 7.14MB/s]
Loaded examples: 3000
Examples missing EOS: 0
Min length: 70
Max length: 430
Average length: 162.93933333333334
First example preview:
"0\n\nQuestion: Tamara is 3 times Kim's height less 4 inches. Tamara and Kim have a combined height of 92 inches. How many inches tall is Tamara?\n\nReasoning:\nLet K = Kim's height\nTamara = 3K - 4\nK + 3K - 4 = 92\n4K - 4 = 92\n4K = 96\nKim = <<24=24>>24 inches\nTamara = (3 * 24) - 4 = <<(3*24)-4=68>>68 inche"
EOS token id: 50256
Last 20 tokens: [1731, 13219, 19, 28, 3104, 4211, 3104, 8331, 198, 42061, 3301, 318, 8257, 8331, 7331, 13, 198, 4242, 8257, 50256]
'hes tall.\n#### 68<|endoftext|>'
' per fruit\n#### 2<|endoftext|>'
'>40 each.\n#### 40<|endoftext|>'
0

Question: Tamara is 3 times Kim's height less 4 inches. Tamara and Kim ha

In [14]:
# 5. SFT 모델 dev 평가
SFT_CKPT = '9_10-1e-05-reasoning.pt'  # epochs=10 기준
!python eval_sft.py --checkpoint {SFT_CKPT} --use_gpu

import json
with open(f'outputs/9_10-1e-05-reasoning_metrics.json') as f:
    print('=== CoT-SFT ===', json.dumps(json.load(f), indent=2))

Loading weights: 100% 148/148 [00:00<00:00, 2531.88it/s, Materializing param=wte.weight]             
GPT2Model LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Prompt length: 47, available generation tokens: 977
EOS generated at length 126
Prompt length: 47 Final length: 126 Generated tokens: 79
Prompt length: 53, available generation tokens: 971
EOS generated at length 137
Prompt length: 53 Final length: 137 Generated tokens: 84
Prompt length: 49, available generation tokens: 975
EOS generated at length 151
Prompt length: 49 Final length: 151 Generated tokens: 102
Prompt length: 46, available generation tokens: 978
Prompt length: 46 Final length: 302 Generated tokens: 256
Prompt length: 47, available generation tokens: 977
EOS generated at length 125
Prompt length: 47 Fina

In [15]:
# 6. DPO rejected 데이터 생성 (SFT 오답만)
!python generate_rejected.py --checkpoint {SFT_CKPT} --use_gpu

with open('data/dpo_pairs.jsonl') as f:
    n = sum(1 for _ in f)
print(f'DPO pairs: {n}')

Loading weights: 100% 148/148 [00:00<00:00, 1267.29it/s, Materializing param=wte.weight]             
GPT2Model LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
generating rejected:   0% 0/1500 [00:00<?, ?it/s]Prompt length: 63, available generation tokens: 961
EOS generated at length 114
Prompt length: 63 Final length: 114 Generated tokens: 51
generating rejected:   0% 1/1500 [00:01<30:47,  1.23s/it]Prompt length: 66, available generation tokens: 958
EOS generated at length 155
Prompt length: 66 Final length: 155 Generated tokens: 89
generating rejected:   0% 2/1500 [00:02<35:14,  1.41s/it]Prompt length: 34, available generation tokens: 990
EOS generated at length 92
Prompt length: 34 Final length: 92 Generated tokens: 58
generating rejected:   0% 3/1500 [00:03<27:37,  1.11

In [7]:
# 7. DPO 학습
SFT_CKPT = '9_10-1e-05-reasoning.pt'

!python train_dpo.py \
    --sft_checkpoint {SFT_CKPT} \
    --use_gpu \
    --epochs 3 \
    --batch_size 4 \
    --lr 1e-6 \
    --beta 0.1

tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 87.5kB/s]
vocab.json: 100% 1.04M/1.04M [00:00<00:00, 8.19MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 8.85MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 6.12MB/s]
config.json: 100% 665/665 [00:00<00:00, 4.18MB/s]
model.safetensors: 100% 548M/548M [00:03<00:00, 177MB/s]
Loading weights: 100% 148/148 [00:00<00:00, 946.17it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
generation_config.json: 100% 124/124 [00:00<00:00, 584kB/s]
Loading weights: 100% 148/148 [00:00<00:00, 1280.48it/s, Materializing param=transformer.wte.weight]            
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+---

In [8]:
# 8. DPO 모델 dev 평가
!python eval_sft.py --checkpoint outputs/dpo_gpt2_model --use_gpu

import json
with open('outputs/dpo_gpt2_model_metrics.json') as f:
    print('=== DPO ===', json.dumps(json.load(f), indent=2))

Loading weights: 100% 148/148 [00:00<00:00, 1065.57it/s, Materializing param=transformer.wte.weight]            
{
  "n": 500,
  "exact_accuracy": 0.008,
  "no_answer_rate": 0.314,
  "format_valid_rate": 0.004,
  "repetition_rate": 0.344,
  "checkpoint": "outputs/dpo_gpt2_model"
}
Metrics     -> outputs/dpo_gpt2_model_metrics.json
Generations -> outputs/dpo_gpt2_model_generations.txt
=== DPO === {
  "n": 500,
  "exact_accuracy": 0.008,
  "no_answer_rate": 0.314,
  "format_valid_rate": 0.004,
  "repetition_rate": 0.344,
  "checkpoint": "outputs/dpo_gpt2_model"
}


In [9]:
# 9. 세 모델 결과 비교
import json, os

result_files = {
    'Base GPT-2': 'outputs/baseline_gpt2_metrics.json',
    'CoT-SFT'   : 'outputs/9_10-1e-05-reasoning_metrics.json',
    'DPO'       : 'outputs/dpo_gpt2_model_metrics.json',
}

print(f"{'Model':<12} {'accuracy':>10} {'format':>10} {'repetition':>12} {'no_answer':>11}")
print('-' * 60)
for name, path in result_files.items():
    if not os.path.exists(path):
        print(f'{name:<12}  (not found)')
        continue
    m = json.load(open(path))
    print(f"{name:<12} {m['exact_accuracy']:>10.1%} {m['format_valid_rate']:>10.1%} "
          f"{m['repetition_rate']:>12.1%} {m['no_answer_rate']:>11.1%}")

Model          accuracy     format   repetition   no_answer
------------------------------------------------------------
Base GPT-2         0.0%       0.0%        48.0%       36.0%
CoT-SFT            1.6%      95.8%         4.0%        0.0%
DPO                0.8%       0.4%        34.4%       31.4%
